In [6]:
import pymysql
import pandas as pd

#1
conn = pymysql.connect(
    host = 'localhost',
    user = 'root',
    password = 'root',
    database = 'ProcessSensorDB',
    charset = 'utf8mb4'
)

#2
equipment_id = 101

#3
query = """
select s.equipment_id, sm.measured_at, s.sensor_name, sm.measured_value
from SensorMeasurement sm
join Sensor s
on sm.sensor_id = s.sensor_id
where s.equipment_id = %s
order by sm.measured_at, s.sensor_name;
"""
df = pd.read_sql(query, conn, params=(equipment_id,))

#4
print("=== 원시 센서 데이터 ===")
print(df.head(10))
print("\n데이터 개수:", len(df))
print("\n센서 목록:")
print(df['sensor_name'].unique())

#5
df['measured_at'] = pd.to_datetime(df['measured_at'])

#6
pivot_df = df.pivot_table(
    index='measured_at',
    columns = 'sensor_name',
    values = 'measured_value',
    aggfunc='mean'
)

#7
pivot_df = pivot_df.sort_index()

#8
pivot_df = pivot_df.reset_index()

#9
print("\n=== 분석용 데이터셋 ===")
print(pivot_df.head(10))

print("\n분석용 데이터셋 크기: ")
print(pivot_df.shape)

#10
file_name = f"EQ{equipment_id}_sensor_data.csv"

pivot_df.to_csv(
    file_name,
    index=False,
    encoding='utf-8-sig'
)

print(f"\nCSV 저장완료: {file_name}")

=== 원시 센서 데이터 ===
   equipment_id         measured_at       sensor_name  measured_value
0           101 2024-03-15 09:05:00  chamber_pressure            1.95
1           101 2024-03-15 09:05:00      chamber_temp          398.00
2           101 2024-03-15 09:15:00  chamber_pressure            2.00
3           101 2024-03-15 09:15:00      chamber_temp          402.00
4           101 2024-03-15 09:25:00  chamber_pressure            2.05
5           101 2024-03-15 09:25:00      chamber_temp          405.00
6           101 2024-03-15 11:15:00  chamber_pressure            1.90
7           101 2024-03-15 11:15:00      chamber_temp          399.00
8           101 2024-03-15 11:30:00  chamber_pressure            2.10
9           101 2024-03-15 11:30:00      chamber_temp          401.00

데이터 개수: 16

센서 목록:
['chamber_pressure' 'chamber_temp']

=== 분석용 데이터셋 ===
sensor_name         measured_at  chamber_pressure  chamber_temp
0           2024-03-15 09:05:00              1.95         398.0
1         

C:\Users\user\AppData\Local\Temp\ipykernel_45192\3404317678.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=(equipment_id,))
